In [1]:
import pandas as pd

In [2]:
train_data = pd.read_csv(r'../Artifacts/train_df.csv')
test_data = pd.read_csv(r'../Artifacts/test_df.csv')

### Normalizing data by MinMax scaling strategy.

In [3]:
print((train_data.groupby('series_id')['maintenance_known'].min()<0).sum())
print((train_data.groupby('series_id')['maintenance_known'].max()>1).sum())

0
0


In [4]:
# 1. Select strictly numeric columns (int and float only)
numeric_features = train_data.select_dtypes(include=['number']).columns.tolist()

# 2. Exclude identifiers, target, and columns scaled separately
exclude_cols = ["series_id", "timestamp", "target", "zone_sin", "zone_cos", "nominal_capacity", "maintenance_known"]
numeric_features = [col for col in numeric_features if col not in exclude_cols]

print("Features to scale:", numeric_features)

Features to scale: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'trend', 'workload_intensity', 'demand_forecast', 'staffing_forecast', 'upstream_quality_forecast', 'promotion_intensity', 'shock_risk', 'unit_reliability_forecast', 'queue_pressure_forecast', 'network_pressure_forecast', 'event_load_forecast', 'service_irregularity_risk_forecast', 'throughput_disruption_risk_forecast']


In [5]:
min_df = train_data.groupby('series_id')[numeric_features].min()
max_df= train_data.groupby('series_id')[numeric_features].max()

In [6]:
min_max_matrix=pd.concat([min_df.add_suffix('_min'),max_df.add_suffix('_max')],axis=1)

In [7]:
import pandas as pd

for col in numeric_features:
    # Convert values that should be numeric
    train_data[col] = pd.to_numeric(train_data[col], errors="coerce")

    # Check for invalid values
    if train_data[col].isna().any():
        raise ValueError(f"{col} contains non-numeric values")

    min_val = train_data[col].min()
    max_val = train_data[col].max()

    # Avoid division by zero for constant columns
    if max_val != min_val:
        train_data[col] = (train_data[col] - min_val) / (max_val - min_val)
    else:
        train_data[col] = 0.0

In [8]:
for col in numeric_features:
    # Scale train
    min_val = train_data['series_id'].map(min_max_matrix[f'{col}_min'])
    max_val = train_data['series_id'].map(min_max_matrix[f'{col}_max'])
    train_data[col] = (train_data[col] - min_val) / (max_val - min_val)

    # Scale test using TRAIN parameters
    min_val_test = test_data['series_id'].map(min_max_matrix[f'{col}_min'])
    max_val_test = test_data['series_id'].map(min_max_matrix[f'{col}_max'])
    test_data[col] = (test_data[col] - min_val_test) / (max_val_test - min_val_test)

In [9]:
nominal_capacity_min=train_data['nominal_capacity'].min()
nominal_capacity_max=train_data['nominal_capacity'].max()
train_data['nominal_capacity'] = (train_data['nominal_capacity'] - nominal_capacity_min) / (nominal_capacity_max - nominal_capacity_min)
test_data['nominal_capacity'] = (test_data['nominal_capacity'] - nominal_capacity_min) / (nominal_capacity_max - nominal_capacity_min)  # Similarly using training set's parameters to normalize test data.

In [10]:
train_data.nunique()

series_id                                  96
timestamp                                3456
hour_sin                                   17
hour_cos                                   19
dow_sin                                     7
dow_cos                                     7
is_weekend                                  2
trend                                    3456
workload_intensity                         48
demand_forecast                          8354
staffing_forecast                       19240
upstream_quality_forecast                8865
promotion_intensity                        27
shock_risk                             292835
maintenance_known                         366
unit_reliability_forecast              308889
queue_pressure_forecast                316236
network_pressure_forecast              327727
event_load_forecast                    273660
service_irregularity_risk_forecast     257771
throughput_disruption_risk_forecast    196765
nominal_capacity                  

In [11]:
from pathlib import Path

filepath = Path("../Artifacts/MinMax_Normalized_Data/train_df_normalized.csv")
filepath.parent.mkdir(parents=True, exist_ok=True)
train_data.to_csv(filepath, index=False)

filepath = Path("../Artifacts/MinMax_Normalized_Data/test_df_normalized.csv")
filepath.parent.mkdir(parents=True, exist_ok=True)
test_data.to_csv(filepath, index=False)

In [ ]:
import numpy as np

df = pd.DataFrame({'nominal_capacity_min': np.repeat(nominal_capacity_min, 96), 'nominal_capacity_max': np.repeat(nominal_capacity_max, 96)})

df


,nominal_capacity_min,nominal_capacity_max
0,56.201276,85.974279
1,56.201276,85.974279
2,56.201276,85.974279
3,56.201276,85.974279
4,56.201276,85.974279
...,...,...
91,56.201276,85.974279
92,56.201276,85.974279
93,56.201276,85.974279
94,56.201276,85.974279


In [13]:
min_max_matrix = pd.concat([min_max_matrix.reset_index(drop=True), 
                            df.reset_index(drop=True)], axis=1)
min_max_matrix

,hour_sin_min,hour_cos_min,dow_sin_min,dow_cos_min,is_weekend_min,trend_min,workload_intensity_min,demand_forecast_min,staffing_forecast_min,upstream_quality_forecast_min,...,promotion_intensity_max,shock_risk_max,unit_reliability_forecast_max,queue_pressure_forecast_max,network_pressure_forecast_max,event_load_forecast_max,service_irregularity_risk_forecast_max,throughput_disruption_risk_forecast_max,nominal_capacity_min,nominal_capacity_max
0,-1.0,-1.0,-0.974928,-0.900969,0.0,-1.731704,0.001633,0.0,-2.479066,0.406849,...,1.941409,0.896609,1.0,21.449244,14.865335,4.912713,0.691960,0.728490,56.201276,85.974279
1,-1.0,-1.0,-0.974928,-0.900969,0.0,-1.731704,0.001633,0.0,-2.479066,0.406849,...,1.941409,1.069718,1.0,14.161617,14.573970,6.095443,0.683321,0.568531,56.201276,85.974279
2,-1.0,-1.0,-0.974928,-0.900969,0.0,-1.731704,0.001633,0.0,-2.479066,0.406849,...,1.941409,0.888185,1.0,15.753022,14.369021,5.368135,0.667405,0.628425,56.201276,85.974279
3,-1.0,-1.0,-0.974928,-0.900969,0.0,-1.731704,0.001633,0.0,-2.479066,0.434544,...,1.941409,0.956566,1.0,13.862975,12.936932,6.423192,0.824994,0.574993,56.201276,85.974279
4,-1.0,-1.0,-0.974928,-0.900969,0.0,-1.731704,0.001633,0.0,-2.479066,0.406849,...,1.941409,0.946997,1.0,16.469461,10.966029,6.243909,0.698819,0.636298,56.201276,85.974279
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,-1.0,-1.0,-0.974928,-0.900969,0.0,-1.731704,0.001633,0.0,-2.460020,0.406849,...,1.941409,0.881878,1.0,17.456254,17.842888,5.166857,0.700815,0.708275,56.201276,85.974279
92,-1.0,-1.0,-0.974928,-0.900969,0.0,-1.731704,0.001633,0.0,-2.479066,0.406849,...,1.941409,0.876370,1.0,20.729131,17.458417,5.801941,0.684633,0.578926,56.201276,85.974279
93,-1.0,-1.0,-0.974928,-0.900969,0.0,-1.731704,0.001633,0.0,-2.479066,0.406849,...,1.941409,0.936034,1.0,18.164718,17.543392,5.473334,0.661189,0.631384,56.201276,85.974279
94,-1.0,-1.0,-0.974928,-0.900969,0.0,-1.731704,0.001633,0.0,-2.435213,0.406849,...,1.941409,0.946940,1.0,20.069761,17.426386,4.784558,0.718214,0.646860,56.201276,85.974279


In [15]:
filepath = Path("../Artifacts/MinMax_Normalized_Data/min_max_matrix.csv")
filepath.parent.mkdir(parents=True, exist_ok=True)
min_max_matrix.to_csv(filepath, index=False)